# Planograma OXXO — Backend (Mueble BCO / CF) — Factibilidad horizontal

**Cambio de regla de negocio respecto a la versión anterior:** se confirmó que los
productos altos (ej. refrescos familiares) siempre van en la charola inferior, que
tiene holgura vertical de sobra. Por lo tanto, el modelo matemático **omite
intencionalmente** las restricciones de `ALTO` y `PROFUNDO`.

La heurística ahora valida únicamente:

1. **Capacidad horizontal:** `ancho_usado + ancho_producto <= WIDTH_MAX`.
2. **Variedad por charola:** máximo `MAX_SKUS_POR_CHAROLA = 10` SKUs distintos por
   charola.

La altura de la charola (`EYE_LEVEL_SHELVES`) deja de ser una restricción y pasa a
ser **únicamente un bono en la función objetivo de la Fase 3 (2-opt)**, premiando
productos de alta demanda colocados en charolas "a la altura de los ojos".

Sin `matplotlib` — el dibujo lo hace Streamlit a partir de `output_planograma.csv`.

In [21]:
# ============================================================
# 1) IMPORTS Y CARGA
# ============================================================
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

df = pd.read_csv('Ejemplo.csv')
df.columns = [str(c).strip() for c in df.columns]
print(f"Filas originales: {len(df):,}")


Filas originales: 497,437


In [22]:
# ============================================================
# 2) CONFIGURACION
# ============================================================
MUEBLE_FILTRO         = 'CF'    # se aplica a todos los segmentos

WIDTH_MAX             = 120.0   # ancho maximo por charola (cm)
MAX_SKUS_POR_CHAROLA  = 10      # limite de variedad por charola
EYE_LEVEL_SHELVES     = {1, 2}  # 0-based; solo afecta el objetivo (Fase 3)
SEPARACION_CM         = 0.5     # espacio minimo entre productos (cm)

for col in ('SEGMENTO_ID', 'MUEBLE_ID'):
    if col not in df.columns:
        raise KeyError(f"La columna obligatoria '{col}' no existe en el CSV.")

df_mueble = df[df['MUEBLE_ID'] == MUEBLE_FILTRO].copy()
del df

if df_mueble.empty:
    raise ValueError(f"No hay filas con MUEBLE_ID='{MUEBLE_FILTRO}'.")

segmentos = sorted(df_mueble['SEGMENTO_ID'].dropna().unique().tolist())
print(f"Mueble: {MUEBLE_FILTRO} | Segmentos encontrados ({len(segmentos)}): {segmentos}")


Mueble: CF | Segmentos encontrados (9): ['BCO', 'CLA', 'HOG', 'HRN', 'OFC', 'PTC', 'REC', 'RET', 'SIN']


## Función `procesar_segmento`

Encapsula las fases 1-3 (limpieza, FFD, bin-packing, 2-opt) y la exportación a CSV
para **un segmento dado**. Se llama en el loop de la celda siguiente.

In [23]:
# ============================================================
# 3) CLASE CHAROLA + FUNCIONES DE FASE 1-4
# ============================================================
import time

class Charola:
    __slots__ = ('idx', 'width_max', 'ancho_usado', 'productos')

    def __init__(self, idx, width_max):
        self.idx         = idx
        self.width_max   = width_max
        self.ancho_usado = 0.0
        self.productos   = []

    def espacio_libre(self):
        return self.width_max - self.ancho_usado

    def cabe(self, ancho_total):
        return (self.ancho_usado + ancho_total + SEPARACION_CM <= self.width_max and
                len(self.productos) < MAX_SKUS_POR_CHAROLA)

    def agregar(self, p):
        p['x_inicio']     = self.ancho_usado
        self.ancho_usado += p['ancho_total_cm'] + SEPARACION_CM
        self.productos.append(p)

    def pct_ocupacion(self):
        return self.ancho_usado / self.width_max * 100 if self.width_max else 0.0


def _limpiar(df_seg):
    cols_base = ['SEGMENTO_ID', 'MUEBLE_ID', 'PLANOGRUPO_DESC', 'TAMANO_DESC',
                 'DIRECCION_LEGO_ID', 'UPC_CVE', 'NUM_FRENTES', 'CHAROLA',
                 'UBICACION_BANDEJA', 'ANCHO', 'ITEM_DESC']
    cols_dedup = [c for c in cols_base if c in df_seg.columns]
    df_seg = df_seg.drop_duplicates(subset=cols_dedup).copy()
    for c in ['ANCHO', 'NUM_FRENTES']:
        df_seg[c] = pd.to_numeric(df_seg[c], errors='coerce')
    df_seg = df_seg.dropna(subset=['ANCHO'])
    df_seg = df_seg[df_seg['ANCHO'] > 0]
    df_seg['NUM_FRENTES'] = df_seg['NUM_FRENTES'].fillna(1).clip(lower=1)
    if 'DIRECCION_LEGO_ID' not in df_seg.columns:
        df_seg['DIRECCION_LEGO_ID'] = 'ID'
    df_seg['DIRECCION_LEGO_ID'] = df_seg['DIRECCION_LEGO_ID'].fillna('ID')
    id_col = 'UPC_CVE' if ('UPC_CVE' in df_seg.columns and df_seg['UPC_CVE'].notna().any()) else 'ITEM_DESC'
    agg_dict = {
        'ITEM_DESC':         'first',
        'PLANOGRUPO_DESC':   'first',
        'ANCHO':             'first',
        'NUM_FRENTES':       'median',
        'DIRECCION_LEGO_ID': 'first',
    }
    if id_col != 'UPC_CVE' and 'UPC_CVE' in df_seg.columns:
        agg_dict['UPC_CVE'] = 'first'
    df_seg = df_seg.groupby(id_col, as_index=False).agg(agg_dict)
    return df_seg, id_col


def _ordenar_ffd(df_seg):
    df_seg = df_seg[df_seg['ANCHO'] <= WIDTH_MAX].copy()
    cap_frentes = (WIDTH_MAX // df_seg['ANCHO']).astype(int).clip(lower=1)
    df_seg['NUM_FRENTES']     = np.minimum(df_seg['NUM_FRENTES'].astype(int), cap_frentes)
    df_seg['ancho_total_cm']  = df_seg['ANCHO'] * df_seg['NUM_FRENTES']
    df_seg['score_demanda']   = df_seg['NUM_FRENTES']
    df_seg['PLANOGRUPO_DESC'] = df_seg['PLANOGRUPO_DESC'].fillna("SIN_CATEGORIA").astype(str)
    df_seg['ITEM_DESC']       = df_seg['ITEM_DESC'].fillna("SIN_NOMBRE").astype(str)
    return df_seg.sort_values('ancho_total_cm', ascending=False).reset_index(drop=True)


def _empacar_best_fit(df_sorted):
    """Fase 2 — Best-Fit: coloca cada producto en la charola con menos espacio
    libre restante tras agregarlo (mejor ajuste). Abre charola nueva si no cabe."""
    charolas = []
    for row in df_sorted.itertuples(index=False):
        p = {
            'UPC_CVE':         getattr(row, 'UPC_CVE', None),
            'ITEM_DESC':       row.ITEM_DESC,
            'PLANOGRUPO_DESC': row.PLANOGRUPO_DESC,
            'ANCHO':           float(row.ANCHO),
            'NUM_FRENTES':     int(row.NUM_FRENTES),
            'ancho_total_cm':  float(row.ancho_total_cm),
            'score_demanda':   int(row.score_demanda),
        }
        mejor_charola  = None
        menor_espacio  = float('inf')
        for c in charolas:
            if c.cabe(p['ancho_total_cm']):
                espacio_restante = c.espacio_libre() - p['ancho_total_cm'] - SEPARACION_CM
                if espacio_restante < menor_espacio:
                    menor_espacio = espacio_restante
                    mejor_charola = c
        if mejor_charola is not None:
            mejor_charola.agregar(p)
        else:
            c = Charola(len(charolas), WIDTH_MAX)
            c.agregar(p)
            charolas.append(c)
    return charolas


def _rellenar_frentes(charolas):
    """Fase 2b — Relleno: agrega frentes extra a productos ya colocados para
    aprovechar el espacio libre de cada charola. Prioriza mayor score_demanda.
    No agrega SKUs nuevos ni rompe MAX_SKUS_POR_CHAROLA."""
    for c in charolas:
        cambio = True
        while cambio:
            cambio = False
            libre  = c.espacio_libre()
            if libre < SEPARACION_CM:
                break
            candidatos = sorted(
                enumerate(c.productos),
                key=lambda x: x[1]['score_demanda'],
                reverse=True
            )
            for idx_p, p in candidatos:
                ancho_frente = p['ANCHO']
                if ancho_frente + SEPARACION_CM <= libre:
                    p['NUM_FRENTES']    += 1
                    p['ancho_total_cm'] += ancho_frente
                    p['score_demanda']  += 1
                    c.ancho_usado       += ancho_frente
                    cambio = True
                    break

    for c in charolas:
        x = 0.0
        for p in c.productos:
            p['x_inicio'] = x
            x += p['ancho_total_cm'] + SEPARACION_CM

    return charolas


def _calcular_objetivo(charolas):
    return sum(
        p['score_demanda'] * (1.5 if c.idx in EYE_LEVEL_SHELVES else 1.0)
        for c in charolas for p in c.productos
    )


def _2opt(charolas, max_iter=300):
    n_mejoras = 0
    for _ in range(max_iter):
        mejor_delta, mejor_swap = 0.0, None
        for i in range(len(charolas)):
            for j in range(i + 1, len(charolas)):
                ca, cb = charolas[i], charolas[j]
                fa = 1.5 if ca.idx in EYE_LEVEL_SHELVES else 1.0
                fb = 1.5 if cb.idx in EYE_LEVEL_SHELVES else 1.0
                for ia, pa in enumerate(ca.productos):
                    for ib, pb in enumerate(cb.productos):
                        na = ca.ancho_usado - pa['ancho_total_cm'] + pb['ancho_total_cm']
                        nb = cb.ancho_usado - pb['ancho_total_cm'] + pa['ancho_total_cm']
                        if na > ca.width_max or nb > cb.width_max:
                            continue
                        delta = (pb['score_demanda'] - pa['score_demanda']) * (fa - fb)
                        if delta > mejor_delta + 1e-9:
                            mejor_delta = delta
                            mejor_swap  = (i, ia, j, ib, na, nb)
        if mejor_swap is None:
            break
        i, ia, j, ib, na, nb = mejor_swap
        ca, cb = charolas[i], charolas[j]
        ca.productos[ia], cb.productos[ib] = cb.productos[ib], ca.productos[ia]
        ca.ancho_usado, cb.ancho_usado = na, nb
        n_mejoras += 1
    for c in charolas:
        x = 0.0
        for p in c.productos:
            p['x_inicio'] = x
            x += p['ancho_total_cm'] + SEPARACION_CM
    return charolas, n_mejoras


def procesar_segmento_direccion(df_mueble, segmento, direccion, tamano, mueble, carpeta_salida='.'):
    """Pipeline completo para una combinacion (SEGMENTO_ID, DIRECCION_LEGO_ID, TAMANO_DESC).
    Fases: limpieza -> FFD -> Best-Fit (F2) -> Relleno (F2b) -> 2-opt (F3) -> CSV.
    Retorna (df_out, tiempo_s) o (None, 0.0) si no hay datos."""
    t0 = time.time()

    df_seg = df_mueble[
        (df_mueble['SEGMENTO_ID']       == segmento) &
        (df_mueble['DIRECCION_LEGO_ID'] == direccion) &
        (df_mueble['TAMANO_DESC']       == tamano)
    ].copy()
    if df_seg.empty:
        print(f"  [SKIP] {segmento}/{direccion}/tam={tamano}: sin datos.")
        return None, 0.0

    df_seg, _ = _limpiar(df_seg)
    df_work   = _ordenar_ffd(df_seg)

    # Fase 2 — Best-Fit
    charolas = _empacar_best_fit(df_work)
    ocup_pre_relleno = np.mean([c.pct_ocupacion() for c in charolas]) if charolas else 0.0

    # Fase 2b — Relleno de frentes
    charolas = _rellenar_frentes(charolas)
    ocup_post_relleno = np.mean([c.pct_ocupacion() for c in charolas]) if charolas else 0.0

    # Fase 3 — 2-opt
    z_antes = _calcular_objetivo(charolas)
    charolas, n_mejoras = _2opt(charolas)
    z_despues = _calcular_objetivo(charolas)

    tiempo_s = time.time() - t0

    # Reporte
    tam_label = f"{tamano:g}".replace(".", "_")
    print(f"  {segmento:>6}/{direccion}/tam={tamano:>3} | Productos={len(df_work):>3} | "
          f"Charolas={len(charolas):>2} | "
          f"Ocup. pre={ocup_pre_relleno:5.1f}% -> post={ocup_post_relleno:5.1f}% | "
          f"Obj {z_antes:.0f}->{z_despues:.0f} | Swaps={n_mejoras} | "
          f"Tiempo={tiempo_s:.3f}s")

    etiqueta = "DI" if direccion == "DI" else "IZ"
    registros = [
        {
            'SEGMENTO_ID':         segmento,
            'MUEBLE_ID':           mueble,
            'DIRECCION':           direccion,
            'TAMANO_DESC':         tamano,
            'CHAROLA_ID':          c.idx + 1,
            'UPC_CVE':             p['UPC_CVE'],
            'ITEM_DESC':           p['ITEM_DESC'],
            'PLANOGRUPO_DESC':     p['PLANOGRUPO_DESC'],
            'NUM_FRENTES':         p['NUM_FRENTES'],
            'COORDENADA_X_INICIO': round(p['x_inicio'], 2),
            'ANCHO_DIBUJO_CM':     round(p['ancho_total_cm'], 2),
            'TIEMPO_COMPUTO_S':    round(tiempo_s, 4),
        }
        for c in charolas for p in c.productos
    ]
    df_out = pd.DataFrame(registros)
    df_out.to_csv(
        f"{carpeta_salida}/output_planograma_{segmento}_{etiqueta}_{tam_label}.csv",
        index=False
    )
    return df_out, tiempo_s

print("Funciones cargadas.")

Funciones cargadas.


In [24]:
# ============================================================
# 4) LOOP POR SEGMENTO, DIRECCION Y TAMANO_DESC
#    Un planograma por cada combinacion (SEGMENTO_ID, DIRECCION_LEGO_ID, TAMANO_DESC)
#    Archivos: output_planograma_{SEGMENTO}_{DI|IZ}_{TAMANO}.csv
#              output_planograma_todos.csv  (consolidado)
# ============================================================
CARPETA_SALIDA = '.'

if 'DIRECCION_LEGO_ID' not in df_mueble.columns:
    df_mueble['DIRECCION_LEGO_ID'] = 'ID'
df_mueble['DIRECCION_LEGO_ID'] = df_mueble['DIRECCION_LEGO_ID'].fillna('ID')

combos = (
    df_mueble[['SEGMENTO_ID', 'DIRECCION_LEGO_ID', 'TAMANO_DESC']]
    .drop_duplicates()
    .sort_values(['SEGMENTO_ID', 'DIRECCION_LEGO_ID', 'TAMANO_DESC'])
    .values.tolist()
)

print(f"Combinaciones a procesar ({len(combos)}) para MUEBLE={MUEBLE_FILTRO}:")
for seg, dir_, tam in combos:
    etiqueta  = "DI" if dir_ == "DI" else "IZ"
    tam_label = f"{tam:g}".replace(".", "_")
    print(f"  output_planograma_{seg}_{etiqueta}_{tam_label}.csv")
print()

dfs_todos    = []
resumen_tiempos = []

for seg, dir_, tam in combos:
    df_result, tiempo_s = procesar_segmento_direccion(
        df_mueble, seg, dir_, tam, MUEBLE_FILTRO, CARPETA_SALIDA
    )
    if df_result is not None:
        dfs_todos.append(df_result)
        etiqueta  = "DI" if dir_ == "DI" else "IZ"
        tam_label = f"{tam:g}".replace(".", "_")
        resumen_tiempos.append({
            'planograma': f"{seg}_{etiqueta}_{tam_label}",
            'tiempo_s':   round(tiempo_s, 4),
        })

if dfs_todos:
    df_consolidado = pd.concat(dfs_todos, ignore_index=True)
    ruta = f"{CARPETA_SALIDA}/output_planograma_todos.csv"
    df_consolidado.to_csv(ruta, index=False)
    print(f"\nConsolidado: {ruta} ({len(df_consolidado)} filas)")

print(f"\nTotal planogramas generados: {len(dfs_todos)}")

# ── Resumen de tiempos computacionales ──────────────────────────────────────
if resumen_tiempos:
    df_tiempos = pd.DataFrame(resumen_tiempos)
    total_s    = df_tiempos['tiempo_s'].sum()
    print(f"\n{'='*55}")
    print("  TIEMPO COMPUTACIONAL POR PLANOGRAMA")
    print(f"{'='*55}")
    print(df_tiempos.to_string(index=False))
    print(f"{'─'*55}")
    print(f"  Total: {total_s:.4f} s  ({total_s/60:.2f} min)")
    print(f"  Promedio por planograma: {df_tiempos['tiempo_s'].mean():.4f} s")
    print(f"{'='*55}")

Combinaciones a procesar (171) para MUEBLE=CF:
  output_planograma_BCO_DI_1.csv
  output_planograma_BCO_DI_1_5.csv
  output_planograma_BCO_DI_2.csv
  output_planograma_BCO_DI_2_5.csv
  output_planograma_BCO_DI_3.csv
  output_planograma_BCO_DI_3_5.csv
  output_planograma_BCO_DI_4.csv
  output_planograma_BCO_DI_4_5.csv
  output_planograma_BCO_DI_5.csv
  output_planograma_BCO_DI_5_5.csv
  output_planograma_BCO_DI_6.csv
  output_planograma_BCO_DI_6_5.csv
  output_planograma_BCO_IZ_1.csv
  output_planograma_BCO_IZ_1_5.csv
  output_planograma_BCO_IZ_2.csv
  output_planograma_BCO_IZ_2_5.csv
  output_planograma_BCO_IZ_3.csv
  output_planograma_BCO_IZ_3_5.csv
  output_planograma_BCO_IZ_4.csv
  output_planograma_BCO_IZ_4_5.csv
  output_planograma_BCO_IZ_5.csv
  output_planograma_BCO_IZ_5_5.csv
  output_planograma_BCO_IZ_6.csv
  output_planograma_BCO_IZ_6_5.csv
  output_planograma_CLA_DI_1.csv
  output_planograma_CLA_DI_1_5.csv
  output_planograma_CLA_DI_2.csv
  output_planograma_CLA_DI_2_5.csv
 